# Evolución de los distritos locales de Sonora (2019–2025)

Sonora se divide en **21 distritos electorales locales**, cada uno formado
por secciones electorales. La cartografía del INE se actualiza cada año:
las secciones se subdividen (reseccionamiento) y, cuando hay nueva
distritación, cambian de distrito.

Esta libreta responde: **¿cómo han cambiado los distritos a lo largo de
los años?** Usa los artefactos de `00_preparacion_datos`: las capas
simplificadas `*_simp_{año}.gpkg` para los mapas y
`seccion_distrito_por_anio.csv` para el análisis tabular.

In [ ]:
import json

import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from vothmo.config import FOLDER_DATA

PROCESSED = FOLDER_DATA / "processed"
MAP_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
CENTRO_SONORA = {"lat": 29.3, "lon": -110.9}

piv = pd.read_csv(PROCESSED / "seccion_distrito_por_anio.csv")
secciones = {y: gpd.read_file(PROCESSED / f"secciones_simp_{y}.gpkg")
             for y in MAP_YEARS}
distritos = {y: gpd.read_file(PROCESSED / f"distritos_simp_{y}.gpkg")
             for y in MAP_YEARS}

resumen = (piv.groupby("anio")
           .agg(secciones=("seccion", "nunique"),
                distritos=("distrito_l", "nunique"),
                municipios=("municipio", "nunique")))
resumen

def geojson_ligero(gdf: gpd.GeoDataFrame) -> dict:
    """GeoJSON con solo geometría e id (los atributos van en los datos)."""
    feats = json.loads(gdf.to_json())["features"]
    for i, f in enumerate(feats):
        f["id"] = i
        f["properties"] = {}
    return {"type": "FeatureCollection", "features": feats}


## 1. Los distritos en el tiempo

Mapa coroplético animado: cada distrito local por año, con el padrón
electoral (`pad`) y la lista nominal (`ln`) de su cartografía. Los
colores identifican al distrito; desliza el año para ver cómo se
redibujan.

In [ ]:
# GeoJSON combinado: cada entidad tiene un id único por año, así
# animation_frame dibuja solo las geometrías de ese año.
features, rows = [], []
for year, g in distritos.items():
    feats = json.loads(g.to_json())["features"]
    for (_, r), feat in zip(g.iterrows(), feats):
        key = f"{year}-{int(r['distrito_l'])}"
        feat["id"] = key
        feat["properties"] = {}
        features.append(feat)
        rows.append({"key": key, "anio": year,
                     "distrito": int(r["distrito_l"]),
                     "pad": r["pad"], "ln": r["ln"]})

gjson = {"type": "FeatureCollection", "features": features}
dd = pd.DataFrame(rows)

# Un solo trace con escala continua de pasos discretos: con color
# categórico px crea un trace por distrito y cada uno embebe el
# GeoJSON completo (el archivo de salida explotaría).
cols = px.colors.qualitative.Light24[:21]
escala = []
for i, c in enumerate(cols):
    escala += [[i / 21, c], [(i + 1) / 21, c]]

fig = px.choropleth_map(
    dd, geojson=gjson, locations="key",
    featureidkey="id", color="distrito",
    animation_frame="anio",
    color_continuous_scale=escala,
    range_color=[0.5, 21.5],
    map_style="carto-positron",
    center=CENTRO_SONORA, zoom=4.6,
    hover_data={"distrito": True, "pad": True, "ln": True, "key": False},
    title="Distritos electorales locales de Sonora, 2019–2025",
)
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=650,
                  coloraxis_showscale=False)
fig.show()

## 2. Reseccionamiento: secciones nuevas, removidas y reasignadas

Comparamos años consecutivos por la clave `seccion` (estable en Sonora):
cuántas secciones se conservan, cuántas son nuevas, cuántas desaparecen
y, de las que se conservan, cuántas **cambiaron de distrito**.

In [ ]:
cambios = []
for a, b in zip(MAP_YEARS[:-1], MAP_YEARS[1:]):
    pa = piv[piv["anio"] == a].set_index("seccion")
    pb = piv[piv["anio"] == b].set_index("seccion")
    comunes = pa.index.intersection(pb.index)
    reasig = int((pa.loc[comunes, "distrito_l"]
                  != pb.loc[comunes, "distrito_l"]).sum())
    cambios.append({
        "periodo": f"{a}–{b}",
        "secciones": len(pb),
        "conservadas": len(comunes),
        "nuevas": len(pb.index.difference(pa.index)),
        "removidas": len(pa.index.difference(pb.index)),
        "reasignadas": reasig,
    })

cambios_df = pd.DataFrame(cambios).set_index("periodo")
display(cambios_df)

fig = px.bar(
    cambios_df.reset_index().melt(
        id_vars="periodo",
        value_vars=["nuevas", "removidas", "reasignadas"],
        var_name="evento", value_name="n"),
    x="periodo", y="n", color="evento",
    barmode="group",
    labels={"n": "Secciones", "periodo": "Periodo", "evento": "Evento"},
    title="Cambios de secciones electorales entre años consecutivos",
)
fig.show()

## 3. ¿De qué distrito a qué distrito se movieron las secciones?

Matriz de transición y diagrama de Sankey entre el primer y el último año
disponible. Edita `ANIO_A` / `ANIO_B` para comparar otros pares.

In [ ]:
ANIO_A, ANIO_B = 2019, 2025  # par de años a comparar

pa = piv[piv["anio"] == ANIO_A].set_index("seccion")["distrito_l"]
pb = piv[piv["anio"] == ANIO_B].set_index("seccion")["distrito_l"]
comunes = pa.index.intersection(pb.index)
trans = pd.crosstab(pa.loc[comunes].rename(f"Distrito {ANIO_A}"),
                    pb.loc[comunes].rename(f"Distrito {ANIO_B}"))
display(trans)

fig = px.imshow(
    trans, text_auto=True, aspect="auto",
    color_continuous_scale="Blues",
    title=f"Secciones compartidas por distrito: {ANIO_A} → {ANIO_B}",
)
fig.show()

In [ ]:
flujos = (trans.stack().rename("n").reset_index()
          .rename(columns={f"Distrito {ANIO_A}": "origen",
                           f"Distrito {ANIO_B}": "destino"}))
flujos = flujos[(flujos["n"] > 0) & (flujos["origen"] != flujos["destino"])]

labels = ([f"{ANIO_A} D{d}" for d in sorted(trans.index)]
          + [f"{ANIO_B} D{d}" for d in sorted(trans.columns)])
idx = {lab: i for i, lab in enumerate(labels)}

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=12, thickness=18),
    link=dict(
        source=[idx[f"{ANIO_A} D{o}"] for o in flujos["origen"]],
        target=[idx[f"{ANIO_B} D{d}"] for d in flujos["destino"]],
        value=flujos["n"],
    ),
))
fig.update_layout(
    title=f"Secciones que cambiaron de distrito {ANIO_A} → {ANIO_B} "
          f"({int(flujos['n'].sum())} secciones)",
    height=700,
)
fig.show()

## 4. Mapa de las secciones reasignadas

Secciones de `ANIO_B` que pertenecían a otro distrito en `ANIO_A` (rojo)
y secciones nuevas (gris).

In [ ]:
gb = secciones[ANIO_B].reset_index(drop=True).copy()
gb["dl_a"] = gb["seccion"].map(pa)
gb["estado"] = "mismo distrito"
gb.loc[gb["dl_a"].isna(), "estado"] = "sección nueva"
gb.loc[gb["dl_a"].notna() & (gb["dl_a"] != gb["distrito_l"]),
       "estado"] = "cambió de distrito"
print(gb["estado"].value_counts())

fig = px.choropleth_map(
    gb, geojson=geojson_ligero(gb), locations=gb.index,
    color="estado",
    color_discrete_map={"mismo distrito": "#cfe8f3",
                        "cambió de distrito": "#d62728",
                        "sección nueva": "#9e9e9e"},
    map_style="carto-positron", center=CENTRO_SONORA, zoom=4.6,
    hover_data={"seccion": True, "municipio": True,
                "distrito_l": True, "dl_a": True},
    title=f"Secciones que cambiaron de distrito {ANIO_A} → {ANIO_B}",
)
fig.update_traces(marker_line_width=0.2, marker_line_color="white")
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=650)
fig.show()

## Notas

- La clave `seccion` es estable en Sonora: la mayoría de las secciones se
  conserva entre años consecutivos; el resto se subdivide o fusiona
  (reseccionamiento municipal).
- `pad` y `ln` de la cartografía son cortes administrativos anuales del
  INE; no coinciden exactamente con la lista nominal del día de la
  elección (libreta `02`).
- 2025 corresponde a la nueva distritación local; compárala con
  2019–2024 para ver el efecto del redibujo.
- Los mapas usan las geometrías simplificadas `*_simp` producidas por
  `00_preparacion_datos`.